# Tool Use / Function Calling

How a model decides to call a tool, how the call is generated and executed, and how results flow back into the loop.

This notebook is an original tutorial (rewritten for depth, 2026 practice).
Read the concept, run the example, complete the exercises.

## Learning Objectives

- Write a tool schema (JSON Schema) and explain how the description drives model selection of tools.
- Trace the full loop: model emits a tool call → orchestrator executes → result returned as an observation → model continues.
- Handle parallel tool calls, invalid arguments, timeouts, and errors-as-observations.
- Explain strict/structured tool schemas and why they matter.

## Mental Model

The model never executes anything. Tool use is a **conversation protocol**:
the model emits a structured *request* ("call `get_invoice` with
`{id: "1234"}`"), your orchestration code executes it, and the *result goes
back into the conversation as a message* — an observation the model reads
before deciding its next step. The loop is:

```
messages ──▶ model ──▶ tool_call(s)? ──yes──▶ execute ──▶ append results ──▶ model ...
                          │ no
                          ▼
                     final answer
```

Everything hard about tool use lives at the two boundaries: the schema (does
the model produce valid, well-chosen calls?) and the result formatting (does
the model correctly interpret what came back, including errors?).

## Important Concepts

- **Tool schema**: name, description, JSON Schema parameters. The *description* is prompt engineering — it is the main lever for when the model picks the tool. Vague descriptions cause wrong-tool and missed-tool failures.
- **Strict schemas**: providers can grammar-constrain tool arguments (OpenAI `strict: true`; Anthropic strict tool use) so arguments always parse — but valid shape ≠ correct values (see notebook 24).
- **Parallel tool calls**: models can emit several independent calls in one turn; execute concurrently, return all results keyed by call id. Order of results must not matter.
- **Errors as observations**: a failed tool call should return a structured error *to the model* ("rate limited, retry after 5s"), not raise in your code — the model can often recover (retry, different args, different tool).
- **Idempotency & side effects**: retried or replayed calls must not double-execute payments/emails; read-only vs mutating tools deserve different permission levels.
- **MCP**: the 2026 interoperability standard for tools — tool servers expose schemas over a protocol instead of being compiled into your app (notebook 28 covers this in depth).

## Need-To-Know Coverage Checklist

- [x] Tool schema anatomy and description-driven selection.
- [x] The full request → execute → observation loop, in code.
- [x] Parallel tool calls with call ids.
- [x] Error handling: validation failures, timeouts, errors-as-observations.
- [x] Strict schema mode and its limits.
- [x] Safety: permissions, mutating vs read-only tools, human approval gates (see notebook 13/18).
- [x] MCP as the tool-interoperability layer (deep dive: notebook 28).

## Deep Study Notes

### Schemas are prompts

The model chooses tools by reading names, descriptions, and parameter
descriptions. "Gets data" loses to "Look up a customer's invoice by invoice
id; use when the user references a specific invoice number." Include: when to
use it, when *not* to, argument formats, and units. Most wrong-tool bugs are
description bugs.

### The loop, precisely

1. Send messages + tool schemas.
2. Model responds with either text (done) or one or more tool calls, each
   with a unique `call_id` and JSON arguments.
3. Orchestrator validates arguments (schema + business rules), executes —
   concurrently if multiple — and appends one result message per `call_id`.
4. Go to 2. Bound the loop (max steps, budget) — models can tool-loop forever.

### Error handling patterns

- **Validation failure**: return the validation error as the tool result;
  the model corrects arguments on the next turn (same re-ask pattern as
  structured output).
- **Execution failure**: catch, classify (retryable? user-facing?), return a
  structured error observation with guidance ("retry", "ask user").
- **Timeouts**: per-tool budgets; return "timed out" as an observation rather
  than hanging the loop.
- **The failure interviewers probe**: raising an exception *out of the loop*
  on tool failure. That turns a recoverable situation into a dead session.

### Safety

Separate read-only from mutating tools; mutating tools get argument
validation, idempotency keys, and (for consequential actions) a human
approval interrupt. Least privilege per agent: a research agent gets search
tools only. Tool results are *untrusted input* — retrieved web content can
carry prompt injection (notebook 18/28).

## Common Failure Modes

- Vague tool descriptions → model picks the wrong tool or answers from memory instead of calling.
- Exceptions raised out of the loop on tool failure → dead session instead of model-driven recovery.
- Results returned without call ids under parallel calls → model misattributes observations.
- Unbounded tool loops → runaway cost; always cap steps.
- Mutating tools without idempotency → a retry double-charges.
- Giant raw tool outputs dumped into context → token blowup; truncate/summarize results.

## Implementation Notes

- Validate arguments *before* executing, even with strict mode on (business rules aren't in the schema).
- Log every call: tool, args, latency, result size, error class — tool telemetry is your best agent-debugging signal.
- Return compact, structured results; the model reads them like any other text.
- Design result formats for the model: field names it can cite, explicit units, explicit error fields.

In [ ]:
"""The full tool-use loop with parallel calls and errors-as-observations.

A scripted fake model demonstrates the exact message protocol; swap
`fake_model` for a real API client and the loop is production-shaped.
"""
import json

TOOLS = {
    "get_invoice": {
        "description": "Look up an invoice by id. Use when the user references a specific invoice number.",
        "execute": lambda args: {"id": args["id"], "amount": 420, "status": "overdue"},
        "validate": lambda args: None if isinstance(args.get("id"), str) else "id must be a string",
    },
    "get_customer": {
        "description": "Look up a customer profile by email.",
        "execute": lambda args: (_ for _ in ()).throw(TimeoutError("upstream timeout")),
        "validate": lambda args: None,
    },
}

SCRIPT = [  # what the fake model does each turn
    {"tool_calls": [  # turn 1: two PARALLEL calls
        {"call_id": "c1", "tool": "get_invoice", "args": {"id": "INV-9"}},
        {"call_id": "c2", "tool": "get_customer", "args": {"email": "a@b.co"}},
    ]},
    {"text": "Invoice INV-9 is $420 overdue. (Customer lookup failed upstream; "
             "I proceeded with invoice data only.)"},  # turn 2: recovers from the error
]


def fake_model(messages, turn):
    return SCRIPT[turn]


def execute_call(call):
    """Validate, execute, and ALWAYS return an observation — never raise."""
    tool = TOOLS[call["tool"]]
    if (err := tool["validate"](call["args"])):
        return {"call_id": call["call_id"], "error": f"invalid args: {err}"}
    try:
        return {"call_id": call["call_id"], "result": tool["execute"](call["args"])}
    except Exception as e:
        return {"call_id": call["call_id"],
                "error": f"{type(e).__name__}: {e}", "retryable": True}


def run_agent(user_msg, max_steps=5):
    messages = [{"role": "user", "content": user_msg}]
    for turn in range(max_steps):
        reply = fake_model(messages, turn)
        if "text" in reply:
            return reply["text"]
        results = [execute_call(c) for c in reply["tool_calls"]]  # parallel in prod
        messages.append({"role": "tool_results", "content": results})
        print(f"turn {turn}: executed {len(results)} calls ->")
        for r in results:
            print("   ", json.dumps(r))
    return "step budget exhausted"


print(run_agent("Is invoice INV-9 paid? Also pull up the customer."))


## Practice

1. Add a `send_refund` tool. Give it an idempotency-key argument and explain
   in a comment why the loop must pass the *same* key on a retry.
2. Make `execute_call` enforce a per-tool timeout budget (simulate with a
   `duration` field) and return "timed out" as an observation.
3. Write two descriptions for `get_invoice` — one that would cause the model
   to also call it for "how do refunds work?" and one that wouldn't.
4. The model emits the same failing call three turns in a row. Where do you
   break the loop, and what do you return to the user?

## Design Checklist

- [ ] Every tool description says when to use it and when not to.
- [ ] Arguments validated before execution; validation errors go back as observations.
- [ ] Parallel calls executed concurrently, results keyed by call id.
- [ ] No exception escapes the loop; every failure becomes a structured observation.
- [ ] Loop bounded by steps and budget.
- [ ] Mutating tools: idempotency keys + approval gates; read-only tools separated.
- [ ] Tool telemetry logged (args, latency, error class).